Collect imbalanced reactions from all models without MEMOTE

Imports

In [1]:
import cobra
from cobra.io import read_sbml_model
from cobra import Reaction
import os
import pandas as pd
from cobra.util.solver import solvers

Paths

In [ ]:
model_path = "/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp_mb2_lib/"

save_path = "/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp_mb2_lib_bz/EEGCs_involved_rxns"


Set Solver

In [3]:
cobra.Configuration().solver = "cplex"

All metabolites to be tested for EEGCs

In [4]:
energy_metabolites = ['atp_c', 'ctp_c', 'gtp_c', 'utp_c', 'itp_c', 'nadph_c', 'nadh_c', 'fadh2_c', 'fmnh2_c', 'q8h2_c', 'mql8_c', 'mql6_c', 'mql7_c', '2dmmql8_c', 'accoa_c', 'glu__L_c']

Energy couples for all tested metabolites as dict

In [5]:
ENERGY_COUPLES = {
    "atp_c": "adp_c",
    "ctp_c": "cdp_c",
    "gtp_c": "gdp_c",
    "utp_c": "udp_c",
    "itp_c": "idp_c",
    "nadph_c": "nadp_c",
    'nadh_c': 'nad_c',
    "fadh2_c": "fad_c",
    "fmnh2_c": "fmn_c",
    "q8h2_c": "q8_c",
    "mql8_c": "mqn8_c",
    "mql6_c": "mqn6_c",
    "mql7_c": "mqn7_c",
    "2dmmql8_c": "2dmmq8_c",
    "accoa_c": "coa_c",
    "glu__L_c": "akg_c"
}

Functions

In [6]:
def close_boundaries_sensibly(model):
    """
    Return a cobra model with all boundaries closed and changed constraints.

    In the returned model previously fixed reactions are no longer constrained
    as such. Instead reactions are constrained according to their
    reversibility. This is to prevent the FBA from becoming infeasible when
    trying to solve a model with closed exchanges and one fixed reaction.

    Parameters
    ----------
    model : cobra.Model
        The metabolic model under investigation.

    Returns
    -------
    cobra.Model
        A cobra model with all boundary reactions closed and the constraints
        of each reaction set according to their reversibility.
    """
    
    for rxn in model.reactions:
        if rxn.reversibility:
            rxn.bounds = -1, 1
        else:
            rxn.bounds = 0, 1
    for boundary in model.boundary:
        boundary.bounds = (0, 0)
    return model

In [7]:
def detect_energy_generating_cycles(model, metabolite_id):

    dissipation_rxn = Reaction("Dissipation")
    if metabolite_id in ["atp_c", "ctp_c", "gtp_c", "utp_c", "itp_c"]:
        # build nucleotide-type dissipation reaction
        dissipation_rxn.add_metabolites(
            {
                model.metabolites.get_by_id('h2o_c'): -1,
                model.metabolites.get_by_id('h_c'): 1,
                model.metabolites.get_by_id('pi_c'): 1,
            }
        )
    elif metabolite_id in ["nadph_c", "nadh_c"]:
        # build nicotinamide-type dissipation reaction
        dissipation_rxn.add_metabolites(
            {model.metabolites.get_by_id('h_c'): 1}
        )
    elif metabolite_id in [
        "fadh2_c",
        "fmnh2_c",
        "q8h2_c",
        "mql8_c",
        "mql6_c",
        "mql7_c",
        "2dmmql8_c"
    ]:
        # build redox-partner-type dissipation reaction
        dissipation_rxn.add_metabolites(
            {model.metabolites.get_by_id('h_c'): 2}
        )
    elif metabolite_id == "accoa_c":
        dissipation_rxn.add_metabolites(
            {
                model.metabolites.get_by_id('h2o_c'): -1,
                model.metabolites.get_by_id('h_c'): 1,
                model.metabolites.get_by_id('ac_c'): 1
            }
        )
    elif metabolite_id == "glu__L_c":
        dissipation_rxn.add_metabolites(
            {
                model.metabolites.get_by_id('h2o_c'): -1,
                model.metabolites.get_by_id('h_c'): 2,
                model.metabolites.get_by_id('nh4_c'): 1
            }
        )

    dissipation_product = ENERGY_COUPLES[metabolite_id]
    dissipation_rxn.add_metabolites({model.metabolites.get_by_id(metabolite_id): -1, model.metabolites.get_by_id(dissipation_product): 1})
    model = close_boundaries_sensibly(model)
    model.add_reactions([dissipation_rxn])
    print(model.reactions.get_by_id("Dissipation").reaction)
    print(model.reactions.get_by_id("Dissipation").bounds)
    model.objective = dissipation_rxn
    print(model.objective.expression)
    solution = model.optimize(raise_error=True)
    print(solution)
    if solution.objective_value > 0.0:
        return (
            solution.fluxes[solution.fluxes.abs() != 0.0]
            .index.drop(["Dissipation"])
            .tolist()
        )
    else:
        return []

In [22]:
EGCs = {}
for file in os.listdir(model_path):
    if not file.endswith(('.xml', '.sbml')):
        continue
    model = read_sbml_model(model_path+file)
    EGCs[model.id] = {}
    for metabolite in energy_metabolites:
        EGCs[model.id][metabolite] = []
        with model:
            print(f"Checking metabolite: {metabolite}")
            try:
                result = detect_energy_generating_cycles(model, metabolite)
            except KeyError:
                print(f"Metabolite {metabolite} not found in model.")
                result = f"Metabolite {metabolite} not found in model."
                continue
            if result:
                EGCs[model.id][metabolite] = result

Checking metabolite: atp_c
atp_c + h2o_c --> adp_c + h_c + pi_c
(0.0, 1000.0)
1.0*Dissipation - 1.0*Dissipation_reverse_fc09b
<Solution 1.000 at 0x730f09d6afb0>
Checking metabolite: ctp_c
ctp_c + h2o_c --> cdp_c + h_c + pi_c
(0.0, 1000.0)
1.0*Dissipation - 1.0*Dissipation_reverse_fc09b
<Solution 1.000 at 0x730f09d6acb0>
Checking metabolite: gtp_c
gtp_c + h2o_c --> gdp_c + h_c + pi_c
(0.0, 1000.0)
1.0*Dissipation - 1.0*Dissipation_reverse_fc09b
<Solution 1.000 at 0x730f09d685e0>
Checking metabolite: utp_c
h2o_c + utp_c --> h_c + pi_c + udp_c
(0.0, 1000.0)
1.0*Dissipation - 1.0*Dissipation_reverse_fc09b
<Solution 1.000 at 0x730f09d69cf0>
Checking metabolite: itp_c
h2o_c + itp_c --> h_c + idp_c + pi_c
(0.0, 1000.0)
1.0*Dissipation - 1.0*Dissipation_reverse_fc09b
<Solution 1.000 at 0x730f09d68c40>
Checking metabolite: nadph_c
nadph_c --> h_c + nadp_c
(0.0, 1000.0)
1.0*Dissipation - 1.0*Dissipation_reverse_fc09b
<Solution 1.000 at 0x730f09d692d0>
Checking metabolite: nadh_c
nadh_c --> h_c +

In [ ]:
for key, value in EGCs['modified_Root275.xml'].items():
    print(f'{key}: {len(value)}')

In [23]:
EGCs_df = pd.DataFrame.from_dict(EGCs)
EGCs_df.to_csv(save_path + '/EEGCs_all.csv')
print(EGCs_df)

                                                      m_939_  \
atp_c      [ACOAD7, ACOAD7f, ALAR, ATPS4rpp, CAT, CYTBD2p...   
ctp_c      [ACOAD7, ACOAD7f, ALAR, ATPS4rpp, CAT, CYTBD2p...   
gtp_c      [ACOAD7, ACOAD7f, ADK1, ADK3, ALAR, ATPS4rpp, ...   
utp_c      [ADK1, ADK3, ALAR, ATPS4rpp, CAT, CYTBD2pp, DL...   
itp_c      [ADK1, ADK4, ALAR, ATPS4rpp, CAT, CYTBD2pp, DL...   
nadph_c    [ADK1, ADK4, GCDH, GLUTCOADHc, NDPK9, PCNO, PP...   
nadh_c     [5DGLCNR, 5DKGR, A6PAG, AADa, AADb, ACCOAL, AC...   
fadh2_c    [A6PAG, AADa, AADb, ACCOAL, ACOAD3, ACOAD3f, A...   
fmnh2_c    [A6PAG, AADa, AADb, ACCOAL, ACOAD3, ACOAD3f, A...   
q8h2_c     [12DGR160tipp, ACCOAL, ACOAD3, ACOAD3f, ACOAD5...   
mql8_c     [12DGR160tipp, AADa, AADb, ACCOAL, ACOAD3, ACO...   
mql6_c                                                    []   
mql7_c     [12DGR161tipp, ACCOAL, ACOAD3, ACOAD3f, ACOAD5...   
2dmmql8_c  [AADa, AADb, ACCOAL, ACOAD2, ACOAD2f, ACOAD3, ...   
accoa_c    [AADa, AADb, ACALD, ACCOAL, A

In [30]:
from collections import defaultdict

def accumulate_egc_results(egcs_dict):
    """
    Accumulates all EGC results across all models into a single flat dictionary.
    
    Parameters
    ----------
    egcs_dict : dict
        The nested dictionary structured as EGCs[model_id][metabolite] = [reactions]
        
    Returns
    -------
    dict
        A flat dictionary structured as {reaction_id: [metabolites]}
    """
    # Use a set to automatically handle duplicates across different models
    accumulated_rxns = defaultdict(set)
    
    for model_id, met_data in egcs_dict.items():
        for metabolite_id, rxn_list in met_data.items():
            # Skip error strings or empty results
            if isinstance(rxn_list, str) or not rxn_list:
                continue
                
            for rxn_id in rxn_list:
                accumulated_rxns[rxn_id].add(metabolite_id)
                
    # Convert sets back to sorted lists for clean reading
    return {rxn_id: sorted(list(mets)) for rxn_id, mets in accumulated_rxns.items()}



In [34]:
fEGCs = accumulate_egc_results(EGCs)

In [36]:
clean_fEGCs = {rxn: ", ".join(mets) for rxn, mets in fEGCs.items()}

fEGCs_df = pd.DataFrame(list(clean_fEGCs.items()), columns=['Reaction', 'Metabolites'])

fEGCs_df.to_csv(save_path + '/fEEGCs_all.csv', index=False)
print(fEGCs_df)

        Reaction                                        Metabolites
0         ACOAD7  2dmmql8_c, accoa_c, atp_c, ctp_c, fadh2_c, fmn...
1        ACOAD7f  2dmmql8_c, accoa_c, atp_c, ctp_c, fadh2_c, fmn...
2           ALAR  2dmmql8_c, accoa_c, atp_c, ctp_c, fadh2_c, fmn...
3       ATPS4rpp  2dmmql8_c, accoa_c, atp_c, ctp_c, fadh2_c, fmn...
4            CAT  2dmmql8_c, accoa_c, atp_c, ctp_c, fadh2_c, fmn...
..           ...                                                ...
897     GTHRDHpp                                             mql7_c
898  GTHRDabc2pp                                             mql7_c
899     GLMS_syn  2dmmql8_c, fadh2_c, fmnh2_c, glu__L_c, mql8_c,...
900    ACSPHAC40                                            nadph_c
901          BDH                                            nadph_c

[902 rows x 2 columns]
